Football Player Future G/A Predictor

Importing Modules

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

Reading the Data

In [ ]:
df23 = pd.read_csv('ModelData/football-data_23-24.csv') 
df24 = pd.read_csv('ModelData/football-data_24-25.csv')
df25 = pd.read_csv('ModelData/football-data_25-26.csv')

print(f"2023-2024 shape: {df23.shape}")
print(f"2024-2025 shape: {df24.shape}")
print(f"2025-2026 shape: {df25.shape}")

2023-2024 shape: (2852, 37)
2024-2025 shape: (2854, 267)
2025-2026 shape: (2779, 102)


Preparing and Cleaning the data

In [3]:
base_features = ['Min', '90s', 'Gls', 'Ast', 'G+A', 'xG', 'npxG', 'xAG', 'PrgC', 'PrgP', 'PrgR']

df23_sub = df23[['Player', 'Age', 'Pos'] + base_features]
df23_sub = df23_sub.add_suffix('_23')
df23_sub = df23_sub.rename(columns={'Player_23': 'Player'})

df24_sub = df24[['Player'] + base_features]
df24_sub = df24_sub.add_suffix('_24')
df24_sub = df24_sub.rename(columns={'Player_24': 'Player'})

df25_target = df25[['Player', 'G+A', '90s']].copy()
df25_target = df25_target.rename(columns={'G+A': 'Target_G+A', '90s': '90s_25'})
df25_target['Target_GA_per90'] = df25_target['Target_G+A'] / df25_target['90s_25']

cleandf = df23_sub.merge(df24_sub, on='Player', how='inner').merge(df25_target, on='Player', how='inner')
cleandf = cleandf.drop_duplicates(subset=['Player'])

cleandf = cleandf[(cleandf['90s_24'] >= 10) & (cleandf['90s_23'] >= 5)].copy()
cleandf['Age_current'] = cleandf['Age_23'].astype(float) + 2

for season in ['23', '24']:
    cleandf[f'GA_per90_{season}']  = cleandf[f'G+A_{season}']  / cleandf[f'90s_{season}']
    cleandf[f'xG_per90_{season}']  = cleandf[f'xG_{season}']   / cleandf[f'90s_{season}']
    cleandf[f'xAG_per90_{season}'] = cleandf[f'xAG_{season}']  / cleandf[f'90s_{season}']

cleandf['GA_consistency'] = (cleandf['GA_per90_24'] + cleandf['GA_per90_23']) / 2
cleandf['GA_volatility'] = abs(cleandf['GA_per90_24'] - cleandf['GA_per90_23'])
cleandf['xG_consistency']  = (cleandf['xG_per90_24']  + cleandf['xG_per90_23'])  / 2
cleandf['xG_volatility']   = abs(cleandf['xG_per90_24']  - cleandf['xG_per90_23'])
cleandf['xAG_consistency'] = (cleandf['xAG_per90_24'] + cleandf['xAG_per90_23']) / 2
cleandf['xAG_volatility']  = abs(cleandf['xAG_per90_24'] - cleandf['xAG_per90_23'])

cleandf['GA_trend'] = cleandf['GA_per90_24'] - cleandf['GA_per90_23']
cleandf['xG_trend'] = cleandf['xG_per90_24'] - cleandf['xG_per90_23']
cleandf['xAG_trend'] = cleandf['xAG_per90_24'] - cleandf['xAG_per90_23']

le = LabelEncoder()
cleandf['Pos_encoded'] = le.fit_transform(cleandf['Pos_23'].fillna('Unknown'))

print(f"Players after filter: {cleandf.shape[0]}")


Players after filter: 921


Training the model using the Test Data using the Random Forest Regressor

In [5]:
features = ['Age_current', 'Pos_encoded', '90s_23', '90s_24','GA_per90_23', 'xG_per90_23', 'xAG_per90_23',
'GA_per90_24', 'xG_per90_24', 'xAG_per90_24','GA_consistency', 'GA_volatility', 'GA_trend', 'xG_consistency', 'xG_volatility', 'xG_trend',
'xAG_consistency', 'xAG_volatility', 'xAG_trend','PrgC_24', 'PrgP_24', 'PrgR_24']

X = cleandf[features].fillna(0)
y = cleandf['Target_GA_per90'].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_per90 = rf_model.predict(X_test)

results = cleandf.loc[X_test.index, ['Player', 'Pos_23', 'Target_G+A', '90s_25']].copy()
results['Predicted_GA_per90'] = y_pred_per90
results['Projected_G+A'] = results['Predicted_GA_per90'] * results['90s_25']

mae = mean_absolute_error(results['Target_G+A'], results['Projected_G+A'])
print(f"Model Accuracy (MAE): {mae:.3f} Goals/Assists off on average\n")

top_predictions = results.sort_values(by='Projected_G+A', ascending=False)

print("🏆 Model's Top 10 Predictions for the 25/26 Season (Test Set):")
print(top_predictions.head(10).to_string(index=False))

Model Accuracy (MAE): 1.671 Goals/Assists off on average

🏆 Model's Top 10 Predictions for the 25/26 Season (Test Set):
                    Player Pos_23  Target_G+A  90s_25  Predicted_GA_per90  Projected_G+A
              Ante Budimir     FW          17    29.6            0.903626      26.747336
           Mason Greenwood  MF,FW          21    25.4            0.904299      22.969192
           Serhou Guirassy     FW          17    25.1            0.831413      20.868465
           Vinicius Júnior     FW          20    28.4            0.606464      17.223583
                João Pedro  FW,MF          20    27.6            0.520859      14.375718
          Federico Dimarco     DF          22    29.1            0.454129      13.215160
Andre-Frank Zambo Anguissa     MF           5    14.0            0.927381      12.983337
                Yann Gboho  MF,FW          10    27.9            0.452430      12.622808
            Julián Álvarez  MF,FW          12    21.1            0.591344      

Final Predictions using the Entire Dataset

In [6]:
all_predictions_per90 = rf_model.predict(X)
global_results = cleandf[['Player', 'Pos_23', 'Target_G+A', '90s_25']].copy()

global_results['Predicted_GA_per90'] = all_predictions_per90
global_results['Projected_G+A'] = global_results['Predicted_GA_per90'] * global_results['90s_25']

mae2 = mean_absolute_error(global_results['Target_G+A'], global_results['Projected_G+A'])
print(f"Model Accuracy (MAE2): {mae2:.3f} Goals/Assists off on average\n")

global_top = global_results.sort_values(by='Projected_G+A', ascending=False)

print("🌍 GLOBAL TOP 10 PREDICTIONS (Entire Dataset):")
print(global_top[['Player', 'Pos_23', 'Target_G+A', 'Projected_G+A']].head(10).to_string(index=False))

Model Accuracy (MAE2): 1.277 Goals/Assists off on average

🌍 GLOBAL TOP 10 PREDICTIONS (Entire Dataset):
         Player Pos_23  Target_G+A  Projected_G+A
     Harry Kane     FW          38      31.429559
  Kylian Mbappé     FW          28      27.521128
 Erling Haaland     FW          32      26.941093
   Ante Budimir     FW          17      26.747336
  Michael Olise  FW,MF          33      24.884801
Mason Greenwood  MF,FW          21      22.969192
   Lamine Yamal     FW          27      22.675225
Serhou Guirassy     FW          17      20.868465
    Deniz Undav  FW,MF          24      20.327843
      Luis Díaz     FW          28      20.150753


Testing the Model using the Gradient Bossting Regressor

In [7]:
from sklearn.ensemble import GradientBoostingRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train, y_train)

y_pred_gb_per90 = gb_model.predict(X_test)

test_results_gb = cleandf.loc[X_test.index, ['Player', 'Pos_23', 'Target_G+A', '90s_25']].copy()
test_results_gb['Predicted_GA_per90'] = y_pred_gb_per90
test_results_gb['Projected_G+A'] = test_results_gb['Predicted_GA_per90'] * test_results_gb['90s_25']

gb_mae = mean_absolute_error(test_results_gb['Target_G+A'], test_results_gb['Projected_G+A'])
print(f"Gradient Boosting Accuracy (Test MAE): {gb_mae:.3f} Goals/Assists off on average\n")

all_predictions_gb_per90 = gb_model.predict(X)
global_results_gb = cleandf[['Player', 'Pos_23', 'Target_G+A', '90s_25']].copy()

global_results_gb['Predicted_GA_per90'] = all_predictions_gb_per90
global_results_gb['Projected_G+A'] = global_results_gb['Predicted_GA_per90'] * global_results_gb['90s_25']

global_top_gb = global_results_gb.sort_values(by='Projected_G+A', ascending=False)

print("🚀 GLOBAL TOP 10 PREDICTIONS (Gradient Boosting):")
print(global_top_gb[['Player', 'Pos_23', 'Target_G+A', 'Projected_G+A']].head(10).to_string(index=False))


Gradient Boosting Accuracy (Test MAE): 1.653 Goals/Assists off on average

🚀 GLOBAL TOP 10 PREDICTIONS (Gradient Boosting):
         Player Pos_23  Target_G+A  Projected_G+A
     Harry Kane     FW          38      35.303134
 Erling Haaland     FW          32      29.725923
  Michael Olise  FW,MF          33      29.612402
  Kylian Mbappé     FW          28      27.692110
   Lamine Yamal     FW          27      26.252049
   Ante Budimir     FW          17      23.739803
Bruno Fernandes  MF,FW          27      22.336596
      Luis Díaz     FW          28      21.653571
    Deniz Undav  FW,MF          24      21.427280
Serhou Guirassy     FW          17      19.079176


Testing the Model using the Extreme Gradient Boosting Regressor

In [8]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

# X, y, cleandf, X_train, X_test, y_train, y_test are carried over from previous cells

# Initialize the XGBoost model
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_xgb_per90 = xgb_model.predict(X_test)

# Calculate MAE for the test set
test_results_xgb = cleandf.loc[X_test.index, ['Player', 'Pos_23', 'Target_G+A', '90s_25']].copy()
test_results_xgb['Predicted_GA_per90'] = y_pred_xgb_per90
test_results_xgb['Projected_G+A'] = test_results_xgb['Predicted_GA_per90'] * test_results_xgb['90s_25']

xgb_mae = mean_absolute_error(test_results_xgb['Target_G+A'], test_results_xgb['Projected_G+A'])
print(f"XGBoost Accuracy (Test MAE): {xgb_mae:.3f} Goals/Assists off on average\n")

# Final predictions using the entire dataset
all_predictions_xgb_per90 = xgb_model.predict(X)
global_results_xgb = cleandf[['Player', 'Pos_23', 'Target_G+A', '90s_25']].copy()

global_results_xgb['Predicted_GA_per90'] = all_predictions_xgb_per90
global_results_xgb['Projected_G+A'] = global_results_xgb['Predicted_GA_per90'] * global_results_xgb['90s_25']

global_top_xgb = global_results_xgb.sort_values(by='Projected_G+A', ascending=False)

print("⚡ GLOBAL TOP 10 PREDICTIONS (XGBoost):")
print(global_top_xgb[['Player', 'Pos_23', 'Target_G+A', 'Projected_G+A']].head(10).to_string(index=False))

XGBoost Accuracy (Test MAE): 1.671 Goals/Assists off on average

⚡ GLOBAL TOP 10 PREDICTIONS (XGBoost):
         Player Pos_23  Target_G+A  Projected_G+A
     Harry Kane     FW          38      33.648192
 Erling Haaland     FW          32      30.927722
  Michael Olise  FW,MF          33      30.438170
  Kylian Mbappé     FW          28      27.996697
   Lamine Yamal     FW          27      25.776508
      Luis Díaz     FW          28      25.171577
Serhou Guirassy     FW          17      23.135876
    Deniz Undav  FW,MF          24      21.976690
Bruno Fernandes  MF,FW          27      21.811701
   Ante Budimir     FW          17      21.023847
